# 00 - Environment setup and the SafeDrive dataset

**Run this notebook first.** It prepares everything the other notebooks share.

**The story:** you are the ML engineer at *SafeDrive Insurance* (motor insurance). Over the next notebooks you will build
1. a **claim fraud model** (traditional ML, XGBoost on SageMaker AI), and
2. a **claims assistant** (GenAI: RAG over policy documents, a Bedrock agent, guardrails),

and then deploy, monitor, cost-optimize and secure both.

| Step | What this notebook does |
|---|---|
| 1 | Installs Python packages and checks your AWS credentials |
| 2 | Creates the practice S3 bucket |
| 3 | Creates one IAM role used by all services (`safedrive-practice-role`) |
| 4 | Generates the synthetic dataset and uploads it to S3 |
| 5 | Creates a sample call recording with Amazon Polly |
| 6 | Checks Amazon Bedrock model access |

**Needs:** AWS credentials that can create S3 buckets and IAM roles (an admin-like sandbox user is easiest).
Use a **sandbox account**: the practice role is broad on purpose (notebook 4.3a shows how to tighten it).

## 1. Install packages and check credentials

Install the packages (run once; restart the kernel afterwards if anything was upgraded).

In [ ]:
%pip install -q -r ../requirements.txt

Import the shared helpers. `U` holds configuration and small utilities, `C` holds "catch-up" functions.

In [ ]:
import sys; sys.path.append("..")
import json
import pandas as pd
from common import utils as U, catchup as C

Who am I? This call fails if credentials are missing or expired.

In [ ]:
sts = U.client("sts")
identity = sts.get_caller_identity()
print("Account:", identity["Account"])
print("Caller :", identity["Arn"])
print("Region :", U.REGION)

> The region comes from `AWS_REGION` / `AWS_DEFAULT_REGION` / your CLI profile (default `us-east-1`).
> Pick a region where Amazon Bedrock, S3 Vectors and AgentCore are available (for example `us-east-1` or `us-west-2`).

## 2. Create the practice bucket

All data, model artifacts and the shared `state.json` live in one bucket.

In [ ]:
s3 = U.client("s3")
BUCKET = U.bucket()
print(BUCKET)

Create it if it does not exist. (`us-east-1` must **not** get a `LocationConstraint` - a classic gotcha.)

In [ ]:
existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if BUCKET in existing:
    print("bucket already exists")
elif U.REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET)
else:
    s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": U.REGION})

Tag the bucket so its cost shows up under the project tag (used in notebook 4.2).

In [ ]:
s3.put_bucket_tagging(Bucket=BUCKET, Tagging={"TagSet": U.TAGS})
s3.get_bucket_tagging(Bucket=BUCKET)["TagSet"]

EventBridge notifications let later notebooks react to new objects (retraining, Knowledge Base refresh).

In [ ]:
s3.put_bucket_notification_configuration(
    Bucket=BUCKET, NotificationConfiguration={"EventBridgeConfiguration": {}})
print("EventBridge notifications enabled")

## 3. Create the practice IAM role

One role, trusted by every service we use, keeps the other notebooks short.

- **Trust policy** = *who* may assume the role (the services below).
- **Permissions policy** = *what* the role may do.

In [ ]:
SERVICES = ["sagemaker", "glue", "lambda", "bedrock", "firehose", "states", "events", "scheduler",
            "bedrock-agentcore", "codebuild", "codepipeline", "databrew", "cloudformation"]
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": "sts:AssumeRole",
                   "Principal": {"Service": [f"{s}.amazonaws.com" for s in SERVICES]}}],
}

Create the role (skipped if it already exists).

In [ ]:
iam = U.client("iam")
try:
    iam.create_role(RoleName=U.ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust_policy),
                    Description="SafeDrive MLA-C02 practice role", Tags=U.TAGS)
    print("role created")
except iam.exceptions.EntityAlreadyExistsException:
    iam.update_assume_role_policy(RoleName=U.ROLE_NAME, PolicyDocument=json.dumps(trust_policy))
    print("role already exists (trust policy refreshed)")

The permissions are **deliberately broad** (service-level wildcards) so that practice is not blocked by IAM.
This is *not* least privilege - notebook 4.3a fixes that.

In [ ]:
ACTIONS = ["s3:*", "s3vectors:*", "sagemaker:*", "bedrock:*", "bedrock-agentcore:*", "glue:*", "athena:*",
           "databrew:*", "dynamodb:*", "kinesis:*", "firehose:*", "lambda:*", "logs:*", "cloudwatch:*",
           "xray:*", "ecr:*", "events:*", "scheduler:*", "states:*", "sns:*", "ssm:GetParameter*",
           "ssm:PutParameter", "codebuild:*", "codepipeline:*", "comprehend:*", "textract:*",
           "rekognition:*", "transcribe:*", "polly:*", "kms:Decrypt", "kms:GenerateDataKey*",
           "kms:DescribeKey", "ec2:Describe*", "ec2:CreateNetworkInterface*",
           "ec2:DeleteNetworkInterface*", "aws-marketplace:ViewSubscriptions"]
permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {"Sid": "PracticeBroadAccess", "Effect": "Allow", "Action": ACTIONS, "Resource": "*"},
        {"Sid": "PassItself", "Effect": "Allow", "Action": "iam:PassRole", "Resource": U.role_arn()},
    ],
}

Attach it as an inline policy.

In [ ]:
iam.put_role_policy(RoleName=U.ROLE_NAME, PolicyName="safedrive-practice-permissions",
                    PolicyDocument=json.dumps(permissions))
ROLE = U.role_arn()
print(ROLE)

> A new role needs a few seconds before services can assume it. If a later call says
> *"role cannot be assumed"*, wait 10 seconds and run it again.

## 4. Generate the SafeDrive dataset

The data is synthetic and created with a fixed random seed, so everyone gets the same files.
It is written to `../data/` (ignored by git).

In [ ]:
from common.generate_data import generate
local_dir = generate(f"{U.REPO_ROOT}/data")
print(local_dir)

### Look at the data
**Policies** - one row per policyholder.

In [ ]:
policies = pd.read_csv(f"{local_dir}/policies.csv")
print(policies.shape)
policies.head(3)

**Claims** - one row per claim. `is_fraud` is the label for the fraud model.

In [ ]:
claims = pd.read_csv(f"{local_dir}/claims.csv")
print(claims.shape)
claims.head(3)

Fraud is rare (about 5%). This **class imbalance** matters in notebooks 1.3 and 2.3.

In [ ]:
claims["is_fraud"].value_counts(normalize=True).round(3)

**claims_v2** - a later month with planted problems (nulls, duplicates, outliers, drift). Used for data quality (1.3) and drift (4.1a).

In [ ]:
claims_v2 = pd.read_parquet(f"{local_dir}/claims_v2.parquet")
print(claims_v2.shape)
claims_v2.isna().sum()

**Telematics** - driving events that we will stream through Kinesis.

In [ ]:
pd.read_json(f"{local_dir}/telematics_stream.jsonl", lines=True).head(3)

**Adjuster notes** - free text with (fake) personal data, for PII redaction and text processing.

In [ ]:
notes = pd.read_json(f"{local_dir}/adjuster_notes.jsonl", lines=True)
print(notes.loc[0, "note"])

**Policy documents** - the knowledge source for the RAG assistant. Each has a `.metadata.json` side file.

In [ ]:
import os
sorted(os.listdir(f"{local_dir}/policy_docs"))

**Golden Q&A** (`qa_eval.jsonl`) for evaluating the assistant, and `finetune_qa.jsonl` (with some bad records to find later).

In [ ]:
pd.read_json(f"{local_dir}/qa_eval.jsonl", lines=True).head(3)

**Images** - simple drawings of damaged cars plus a claim form (for Rekognition, Textract and image embeddings).

In [ ]:
from PIL import Image
Image.open(f"{local_dir}/images/damage_00.png").resize((320, 200))

### Upload everything to S3
Target: `s3://<bucket>/safedrive/raw/`

In [ ]:
n = U.upload_dir(local_dir, C.RAW)
print(n, "files uploaded to", U.s3_uri(C.RAW))

Check what is there.

In [ ]:
keys = U.list_keys(C.RAW)
print(len(keys), "objects")
keys[:8]

## 5. A sample call recording (Amazon Polly)

Customers also phone in claims. We create a short audio file with Polly so that notebook 2.1 can transcribe it.

In [ ]:
polly = U.client("polly")
CALL_TEXT = ("Hello, I want to report a claim. My policy number is P 0 0 0 1 2 3. "
             "Yesterday evening another car hit my rear bumper at a traffic signal. "
             "Nobody was hurt and I have filed a police report.")
speech = polly.synthesize_speech(Text=CALL_TEXT, OutputFormat="mp3", VoiceId="Joanna")

Save the audio to S3 next to the other raw data.

In [ ]:
AUDIO_KEY = f"{C.RAW}/audio/call_sample.mp3"
s3.put_object(Bucket=BUCKET, Key=AUDIO_KEY, Body=speech["AudioStream"].read())
print(U.s3_uri(AUDIO_KEY))

## 6. Save the shared state

Notebooks share names and IDs through `s3://<bucket>/safedrive/state.json`.

In [ ]:
state = U.save_state(raw_prefix=C.RAW, bucket=BUCKET, region=U.REGION, role_arn=ROLE, audio_key=AUDIO_KEY)
state

## 7. Check Amazon Bedrock model access

The notebooks use Amazon Nova (text) and Amazon Titan (embeddings) by default.
Change them with the `SAFEDRIVE_*_MODEL` environment variables (see `common/utils.py`).

In [ ]:
print("text  :", U.TEXT_MODEL)
print("small :", U.SMALL_MODEL)
print("large :", U.LARGE_MODEL)
print("embed :", U.EMBED_MODEL)

List a few foundation models available in this region.

In [ ]:
bedrock = U.client("bedrock")
models = bedrock.list_foundation_models()["modelSummaries"]
pd.DataFrame(models)[["modelId", "providerName", "inputModalities", "outputModalities"]].head(10)

Try each text model with one small request. `OK` means you have access.

In [ ]:
for model_id in [U.SMALL_MODEL, U.TEXT_MODEL, U.LARGE_MODEL]:
    try:
        answer = U.converse("Reply with the single word OK.", model_id=model_id, max_tokens=10)
        print(f"{model_id:35s} -> {answer.strip()}")
    except Exception as e:
        print(f"{model_id:35s} -> NO ACCESS: {str(e)[:120]}")

Try the embedding model. You should see a vector length of 1024.

In [ ]:
try:
    print("embedding length:", len(U.embed("windscreen damage claim")))
except Exception as e:
    print("NO ACCESS:", str(e)[:150])

> **If a model shows NO ACCESS:** check the model ID and region, and the *Model access* / model catalog page of
> the Amazon Bedrock console. Models with an ID prefix such as `us.` are **cross-Region inference profiles**.

## Done

| Created | Name |
|---|---|
| S3 bucket | `safedrive-mla-<account>-<region>` |
| IAM role | `safedrive-practice-role` |
| Raw data | `s3://<bucket>/safedrive/raw/` |
| State file | `s3://<bucket>/safedrive/state.json` |

Nothing created here has a running cost (only a few MB of S3 storage).

**Next:** `domain1_data_preparation/1.1_collect_store_data.ipynb`.
When you have finished all practice, run `99_cleanup/99_teardown.ipynb`.